In [1]:
%pip install pandas pyyaml ruff

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1
import logging
from pathlib import Path
from datetime import datetime
# Create logs folder
Path("logs").mkdir(exist_ok=True)
# Create timestamp for this run
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.StreamHandler()
    ],
    force=True
)
logger = logging.getLogger("DataPipeline")
logger.info("Data pipeline started")
logger.warning("Warning logging is enabled")
logger.error("Error logging is enabled")

2026-11-23 12:36:08,865 - INFO - Data pipeline started
2026-11-23 12:36:08,866 - WARNING - Warning logging is enabled
2026-11-23 12:36:08,867 - ERROR - Error logging is enabled


In [ ]:
# 2
log_file = Path("logs") / f"pipeline_{run_timestamp}.log"
file_handler = logging.FileHandler(log_file, encoding="utf-8")
file_handler.setLevel(logging.INFO)
formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(message)s"
)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.info("Timestamped log file created: %s", log_file)

2026-11-23 12:36:50,715 - INFO - Timestamped log file created: logs\pipeline_20261123_123608.log


In [7]:
%%writefile config.yaml

paths:
  input_dir: "."
  output_dir: "output"
  log_dir: "logs"
  manifest: "manifest.json"
  summary: "run_summary.json"

cleaning:
  remove_empty_rows: true
  trim_string_values: true
  remove_duplicates: false

thresholds: {}

mappings: {}

Overwriting config.yaml


In [8]:
import os

print(os.path.exists("config.yaml"))

True


In [9]:
import yaml

with open("config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

input_dir = Path(config["paths"]["input_dir"])
output_dir = Path(config["paths"]["output_dir"])
log_dir = Path(config["paths"]["log_dir"])

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

manifest_path = Path(config["paths"]["manifest"])
summary_path = Path(config["paths"]["summary"])

logger.info("Configuration loaded successfully")
logger.info("Input directory: %s", input_dir)
logger.info("Output directory: %s", output_dir)

2026-11-23 12:42:20,021 - INFO - Configuration loaded successfully
2026-11-23 12:42:20,022 - INFO - Input directory: .
2026-11-23 12:42:20,023 - INFO - Output directory: output


In [10]:
# 4
import pandas as pd

def clean_dataframe(df):

    original_count = len(df)

    if config["cleaning"]["trim_string_values"]:
        string_columns = df.select_dtypes(
            include=["object", "string"]
        ).columns

        for column in string_columns:
            df[column] = df[column].map(
                lambda value: value.strip()
                if isinstance(value, str)
                else value
            )

    if config["cleaning"]["remove_empty_rows"]:
        df = df.dropna(how="all")

    if config["cleaning"]["remove_duplicates"]:
        df = df.drop_duplicates()

    cleaned_count = len(df)

    logger.info(
        "Cleaning completed. Original rows: %d, Cleaned rows: %d",
        original_count,
        cleaned_count
    )

    return df

2026-11-23 12:43:32,979 - INFO - NumExpr defaulting to 12 threads.


In [11]:
def save_output(df, file_name):

    output_name = Path(file_name).stem

    csv_path = output_dir / f"{output_name}_cleaned.csv"
    json_path = output_dir / f"{output_name}_cleaned.json"

    # Save CSV
    df.to_csv(csv_path, index=False)

    # Save JSON
    df.to_json(
        json_path,
        orient="records",
        indent=4,
        date_format="iso"
    )

    logger.info("CSV output saved: %s", csv_path)
    logger.info("JSON output saved: %s", json_path)

    return csv_path, json_path

In [12]:
# 5
import json
import time

run_start_time = time.time()

run_summary = {
    "run_timestamp": run_timestamp,
    "files_discovered": 0,
    "files_processed": 0,
    "files_skipped": 0,
    "total_input_records": 0,
    "total_cleaned_records": 0,
    "total_rejected_records": 0,
    "duration_seconds": 0
}

In [13]:
# 6
import hashlib

def calculate_file_hash(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

In [14]:
def load_manifest():

    if manifest_path.exists():
        with open(manifest_path, "r", encoding="utf-8") as file:
            return json.load(file)

    return {"processed_files": {}}

In [15]:
def save_manifest(manifest):

    with open(manifest_path, "w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=4)

    logger.info("Manifest updated successfully")

In [16]:
def is_already_processed(file_path, manifest):

    file_hash = calculate_file_hash(file_path)

    previous_hash = manifest["processed_files"].get(
        file_path.name
    )

    if previous_hash == file_hash:
        logger.info("Skipping already processed file: %s", file_path.name)
        return True

    return False

In [ ]:
# 7
import os

print(os.path.exists("pipeline.py"))

True


In [19]:
from pipeline import run_pipeline

run_pipeline()

2026-11-23 12:55:31,902 - INFO - Pipeline execution started
2026-11-23 12:55:31,906 - INFO - Pipeline execution completed
2026-11-23 12:55:31,906 - INFO - Run summary: {'run_timestamp': '2026-11-23T12:55:31.904220', 'files_discovered': 0, 'files_processed': 0, 'files_skipped': 0, 'files_failed': 0, 'total_input_records': 0, 'total_cleaned_records': 0, 'total_rejected_records': 0, 'duration_seconds': 0.004}


{'run_timestamp': '2026-11-23T12:55:31.904220',
 'files_discovered': 0,
 'files_processed': 0,
 'files_skipped': 0,
 'files_failed': 0,
 'total_input_records': 0,
 'total_cleaned_records': 0,
 'total_rejected_records': 0,
 'duration_seconds': 0.004}

In [22]:
from pipeline import run_pipeline

summary = run_pipeline("config.yaml")

2026-11-23 12:56:43,094 - INFO - Pipeline execution started
2026-11-23 12:56:43,100 - INFO - Pipeline execution completed
2026-11-23 12:56:43,102 - INFO - Run summary: {'run_timestamp': '2026-11-23T12:56:43.097681', 'files_discovered': 0, 'files_processed': 0, 'files_skipped': 0, 'files_failed': 0, 'total_input_records': 0, 'total_cleaned_records': 0, 'total_rejected_records': 0, 'duration_seconds': 0.009}


In [ ]:
# 7
%%writefile pipeline.py

import json
import time
import hashlib
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import yaml


def load_config(config_path):

    with open(config_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)


def setup_logging(log_dir):

    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"pipeline_{timestamp}.log"

    logger = logging.getLogger("DataPipeline")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    formatter = logging.Formatter(
        "%(asctime)s - %(levelname)s - %(message)s"
    )

    file_handler = logging.FileHandler(
        log_file, encoding="utf-8"
    )
    console_handler = logging.StreamHandler()

    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    return logger


def calculate_hash(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


def load_manifest(manifest_path):

    if manifest_path.exists():
        with open(manifest_path, "r", encoding="utf-8") as file:
            return json.load(file)

    return {"processed_files": {}}


def save_manifest(manifest, manifest_path):

    with open(manifest_path, "w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=4)


def clean_dataframe(df, cleaning_config):

    if cleaning_config.get("trim_string_values", True):

        string_columns = df.select_dtypes(
            include=["object", "string"]
        ).columns

        for column in string_columns:
            df[column] = df[column].map(
                lambda value: value.strip()
                if isinstance(value, str)
                else value
            )

    if cleaning_config.get("remove_empty_rows", True):
        df = df.dropna(how="all")

    if cleaning_config.get("remove_duplicates", False):
        df = df.drop_duplicates()

    return df


def save_output(df, file_path, output_dir):

    output_name = file_path.stem

    csv_path = output_dir / f"{output_name}_cleaned.csv"
    json_path = output_dir / f"{output_name}_cleaned.json"

    df.to_csv(csv_path, index=False)

    df.to_json(
        json_path,
        orient="records",
        indent=4,
        date_format="iso"
    )


def run_pipeline(config_path="config.yaml"):

    start_time = time.time()

    config = load_config(config_path)

    input_dir = Path(config["paths"]["input_dir"])
    output_dir = Path(config["paths"]["output_dir"])
    log_dir = Path(config["paths"]["log_dir"])

    manifest_path = Path(config["paths"]["manifest"])
    summary_path = Path(config["paths"]["summary"])

    output_dir.mkdir(parents=True, exist_ok=True)

    logger = setup_logging(log_dir)

    logger.info("Pipeline execution started")

    manifest = load_manifest(manifest_path)

    csv_files = sorted(input_dir.glob("*.csv"))

    summary = {
        "run_timestamp": datetime.now().isoformat(),
        "files_discovered": len(csv_files),
        "files_processed": 0,
        "files_skipped": 0,
        "files_failed": 0,
        "total_input_records": 0,
        "total_cleaned_records": 0,
        "total_rejected_records": 0,
        "duration_seconds": 0
    }

    for file_path in csv_files:

        try:

            file_hash = calculate_hash(file_path)
            file_key = str(file_path.resolve())

            previous_hash = manifest["processed_files"].get(
                file_key
            )

            if previous_hash == file_hash:

                logger.info(
                    "Skipping previously processed file: %s",
                    file_path.name
                )

                summary["files_skipped"] += 1
                continue

            logger.info("Processing file: %s", file_path.name)

            df = pd.read_csv(file_path)

            input_count = len(df)

            cleaned_df = clean_dataframe(
                df,
                config["cleaning"]
            )

            cleaned_count = len(cleaned_df)

            save_output(
                cleaned_df,
                file_path,
                output_dir
            )

            manifest["processed_files"][file_key] = file_hash

            summary["files_processed"] += 1
            summary["total_input_records"] += input_count
            summary["total_cleaned_records"] += cleaned_count

            logger.info(
                "Successfully processed %s",
                file_path.name
            )

        except Exception:

            summary["files_failed"] += 1
            logger.exception(
                "Failed to process file: %s",
                file_path.name
            )

    save_manifest(manifest, manifest_path)

    summary["duration_seconds"] = round(
        time.time() - start_time, 3
    )

    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=4)

    logger.info("Pipeline execution completed")
    logger.info("Run summary: %s", summary)

    return summary

UsageError: Line magic function `%%writefile` not found.


In [28]:
# 8
import subprocess
import sys
result = subprocess.run(
    [sys.executable, "-m", "ruff", "check", ".", "--output-format", "text"],
    capture_output=True,
    text=True,
    check=False
)

print(result.stdout)

In [30]:
import subprocess
import sys
from pathlib import Path

python_files = [
    str(file)
    for file in Path(".").rglob("*.py")
    if ".venv" not in file.parts
    and "__pycache__" not in file.parts
]

print("Python files found:")
for file in python_files:
    print(file)

result = subprocess.run(
    [sys.executable, "-m", "ruff", "check", *python_files, "--fix"],
    capture_output=True,
    text=True,
    check=False
)

print("\nRuff Results:")
print(result.stdout)
print(result.stderr)

print("Exit code:", result.returncode)

Python files found:
pipeline.py
reader.py
schemas.py
transformer.py
valid.py
validator.py
tests\test_transformer.py

Ruff Results:
All checks passed!


Exit code: 0


In [31]:
result = subprocess.run(
    [sys.executable, "-m", "ruff", "format", *python_files],
    capture_output=True,
    text=True,
    check=False
)

print(result.stdout)

7 files reformatted



In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "ruff", "check", *python_files],
    capture_output=True,
    text=True,
    check=False
)

print(result.stdout)

if result.returncode == 0:
    print("SUCCESS: All Python modules passed Ruff checks!")
else:
    print("Some issues still need manual correction.")

All checks passed!

SUCCESS: All Python modules passed Ruff checks!
